# Barcelona Data Preparation

This notebook prepares the Barcelona AirDNA/Airbnb dataset for the capstone project.

## Final outputs

- `barcelona_listings_clean.csv`  
  - Unit of analysis: **1 row = 1 Airbnb listing**
- `barcelona_monthly_metrics_clean.csv`  
  - Unit of analysis: **1 row = 1 Airbnb listing + 1 month**

The monthly dataset is extracted from the embedded JSON stored in the `months` column of the listings dataset.


## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

## 2. Define project paths

The notebook assumes it is executed from the `notebooks/` folder.

Expected structure:

```text
KPMG_Airbnb_Capstone/
├── data/
│   ├── raw/
│   │   └── barcelona/
│   └── processed/
│       └── barcelona/
└── notebooks/
```


In [2]:
BASE_DIR = Path("..")

RAW_DIR = BASE_DIR / "data" / "raw" / "barcelona"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "barcelona"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory:", RAW_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())

Raw directory: /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/raw/barcelona
Processed directory: /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/processed/barcelona


## 3. Load raw Barcelona listings dataset

This notebook uses the listings dataset as the main raw source because it contains:

- one row per listing,
- listing and host characteristics,
- location variables,
- performance metrics,
- the embedded monthly history in the `months` column.

If your local file has `(1)` in the name, rename it to:

```text
listings_BARCELONA_CONVERT_FROM_PARQUET.csv
```


In [3]:
listings_file = RAW_DIR / "listings_BARCELONA_CONVERT_FROM_PARQUET.csv"

if not listings_file.exists():
    raise FileNotFoundError(
        f"File not found: {listings_file}\n"
        "Please check that the file exists in data/raw/barcelona/ "
        "and rename it to listings_BARCELONA_CONVERT_FROM_PARQUET.csv"
    )

listings = pd.read_csv(listings_file)

print("Listings shape:", listings.shape)

Listings shape: (2594, 81)


In [4]:
listings.head()

,listing_id,host_id,instant_book,professional_management,county,latitude,longitude,guests,bedrooms,listing_type,...,l90d_bookable_days,l90d_occupancy,l90d_adjusted_occupancy,l90d_revpar,l90d_adjusted_revpar,l90d_native_revpar,l90d_native_adjusted_revpar,l90d_avg_rate,l90d_avg_native_rate,months
0,14948804,cb910e924e2d,NaN,NaN,Barcelonès,4140698000,218241000,2.0,1.0,Private room in rental unit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{""date"": 1614556800000, ""available_days"": 31,..."
1,1175393002332902164,b83bba1b76d4,False,False,Barcelonès,4139989000,218471000,NaN,NaN,Private room in rental unit,...,86.0,0.033,0.035,4.4,4.6,3.9,4.1,136.2,120.0,"[{""date"": 1717200000000, ""available_days"": 30,..."
2,1557316112282060644,3a5156e7345f,NaN,False,NaN,4137620000,216820000,2.0,1.0,Room in hotel,...,14.0,0.000,0.000,0.0,0.0,0.0,0.0,73.2,64.5,"[{""date"": 1761955200000, ""available_days"": 30,..."
3,3110443,88535ad25871,NaN,True,Barcelonès,4139186000,217150000,6.0,3.0,Entire rental unit,...,90.0,0.622,0.622,283.5,283.5,249.9,249.9,432.2,380.9,"[{""date"": 1614556800000, ""available_days"": 24,..."
4,3491453,3c15645ab066,NaN,False,Barcelonès,4140890000,217270000,6.0,4.0,Entire rental unit,...,64.0,0.222,0.312,65.9,92.7,58.1,81.6,261.8,230.7,"[{""date"": 1614556800000, ""available_days"": 31,..."


## 4. Initial data audit

Before cleaning, we confirm the unit of analysis and inspect the main structure of the dataset.


In [5]:
print("Rows:", len(listings))
print("Unique listing_id:", listings["listing_id"].nunique())
print("Duplicate listing_id:", listings["listing_id"].duplicated().sum())

Rows: 2594


Unique listing_id: 2594
Duplicate listing_id: 0


Expected interpretation:

- If `Rows` equals `Unique listing_id`
- And `Duplicate listing_id` equals `0`

then **1 row = 1 unique Airbnb listing**.


In [6]:
listings.dtypes.value_counts()

float64    58
str        13
object      6
int64       4
Name: count, dtype: int64

In [7]:
missing = (
    listings.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing.head(30)

instant_book                       2194
native_extra_guest_fee             1325
extra_guest_fee                    1325
professional_management            1239
native_cleaning_fee                1053
cleaning_fee                       1053
registration                        911
cohost                              911
l90d_available_days                 699
l90d_blocked_days                   699
l90d_bookable_days                  699
ttm_days_booked                     699
ttm_total_days                      699
ttm_blocked_days                    699
ttm_available_days                  699
ttm_unavailable_days                699
ttm_revenue                         699
ttm_native_revenue                  699
ttm_avg_rate                        699
ttm_avg_native_rate                 699
ttm_months_with_data                699
l90d_reservations_count             699
l90d_native_revenue                 699
ttm_reservations_count              699
ttm_cleaning_fee_revenue            699


In [8]:
missing_pct = (
    listings.isnull()
    .mean()
    .sort_values(ascending=False)
    * 100
)

missing_pct.head(30)

instant_book                       84.579800
native_extra_guest_fee             51.079414
extra_guest_fee                    51.079414
professional_management            47.764071
native_cleaning_fee                40.593678
cleaning_fee                       40.593678
registration                       35.119507
cohost                             35.119507
l90d_available_days                26.946800
l90d_blocked_days                  26.946800
l90d_bookable_days                 26.946800
ttm_days_booked                    26.946800
ttm_total_days                     26.946800
ttm_blocked_days                   26.946800
ttm_available_days                 26.946800
ttm_unavailable_days               26.946800
ttm_revenue                        26.946800
ttm_native_revenue                 26.946800
ttm_avg_rate                       26.946800
ttm_avg_native_rate                26.946800
ttm_months_with_data               26.946800
l90d_reservations_count            26.946800
l90d_nativ

In [9]:
column_audit = pd.DataFrame({
    "column": listings.columns,
    "dtype": listings.dtypes.values,
    "missing_pct": listings.isnull().mean().values * 100
}).sort_values("missing_pct", ascending=False)

column_audit.head(30)

,column,dtype,missing_pct
2,instant_book,object,84.579800
39,native_extra_guest_fee,float64,51.079414
25,extra_guest_fee,float64,51.079414
3,professional_management,object,47.764071
38,native_cleaning_fee,float64,40.593678
24,cleaning_fee,float64,40.593678
20,registration,object,35.119507
19,cohost,object,35.119507
64,l90d_available_days,float64,26.946800
63,l90d_blocked_days,float64,26.946800


## 5. Geographic variables

For Barcelona:

- `neighborhood` corresponds to the broader district level.
- `subdivision` corresponds to the more granular neighbourhood/barrio level.

For this project, `subdivision` should be the main geographic variable for neighbourhood-level analysis, while `neighborhood` is useful for higher-level aggregation.


In [10]:
print("Neighbourhood-like columns:")
print([c for c in listings.columns if "neigh" in c.lower()])

print("\nSubdivision-like columns:")
print([c for c in listings.columns if "sub" in c.lower()])

Neighbourhood-like columns:
['neighborhood']

Subdivision-like columns:
['subdivision']


In [11]:
listings["neighborhood"].value_counts(dropna=False).head(20)

neighborhood
Eixample                 874
Old Town                 594
Sants-Montjuïc           253
NaN                      240
Sant Martí               200
Gràcia                   168
Sarrià - Sant Gervasi     96
Horta-Guinardó            62
les Corts                 44
Sant Andreu               32
Nou Barris                28
Districte II               3
Name: count, dtype: int64

In [12]:
listings["subdivision"].value_counts(dropna=False).head(20)

subdivision
la Dreta de l'Eixample                   295
NaN                                      240
el Raval                                 203
Sant Pere, Santa Caterina i la Ribera    163
Gothic Quarter                           160
l'Antiga Esquerra de l'Eixample          148
la Sagrada Família                       144
el Poble-sec                             118
la Nova Esquerra de l'Eixample           114
la Vila de Gràcia                        104
Sant Antoni                               79
la Barceloneta                            68
el Fort Pienc                             67
el Poblenou                               59
Sant Gervasi - Galvany                    44
Sants                                     44
el Camp d'en Grassot i Gràcia Nova        37
el Camp de l'Arpa del Clot                35
Sants-Badal                               34
Hostafrancs                               32
Name: count, dtype: int64

## 6. Host audit

This is useful for measuring commercialisation and host concentration.


In [13]:
print("Listings:", listings["listing_id"].nunique())
print("Unique hosts:", listings["host_id"].nunique())

host_listing_counts = (
    listings.groupby("host_id")["listing_id"]
    .nunique()
    .sort_values(ascending=False)
)

host_listing_counts.describe()

Listings: 2594
Unique hosts: 1637


count    1637.000000
mean        1.584606
std         2.257767
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        40.000000
Name: listing_id, dtype: float64

In [14]:
host_listing_counts.head(20)

host_id
37a37e192b54    40
c646962ba23d    36
241661982ace    29
2823a5b38d2a    28
04ae688067d2    24
8db87062e6de    20
2ad0309a059d    17
36f2cf85509f    17
7477d6287387    16
603f3791d348    15
73dc43f89ecf    15
dfbad75b57cf    13
b0590974530c    12
ac2a0a73ace9    12
c0afa5c62b3b    12
32b72fb97894    10
7178215478d8     9
9f5960ba9020     9
19db3d8bfc0a     9
ef284fe23d0a     8
Name: listing_id, dtype: int64

## 7. Select core listing-level columns

The goal is to keep only columns that are useful for the capstone:

- listing identification,
- host identification,
- location,
- property characteristics,
- review/rating information,
- compliance/licensing,
- recent performance metrics,
- embedded monthly history.


In [15]:
core_columns = [
    "listing_id",
    "host_id",

    "professional_management",
    "superhost",
    "cohost",

    "neighborhood",
    "subdivision",

    "latitude",
    "longitude",

    "listing_type",
    "room_type",

    "guests",
    "bedrooms",
    "beds",
    "baths",

    "num_reviews",
    "star_rating",

    "registration",

    "ttm_revenue",
    "ttm_days_booked",
    "ttm_avg_rate",

    "l90d_revenue",
    "l90d_occupancy",
    "l90d_revpar",

    "months"
]

missing_cols = [c for c in core_columns if c not in listings.columns]
print("Missing selected columns:", missing_cols)

Missing selected columns: []


In [16]:
if missing_cols:
    raise ValueError(f"The following selected columns are missing: {missing_cols}")

listings_clean = listings[core_columns].copy()

print("Listings clean shape:", listings_clean.shape)
listings_clean.head()

Listings clean shape: (2594, 25)


,listing_id,host_id,professional_management,superhost,cohost,neighborhood,subdivision,latitude,longitude,listing_type,...,num_reviews,star_rating,registration,ttm_revenue,ttm_days_booked,ttm_avg_rate,l90d_revenue,l90d_occupancy,l90d_revpar,months
0,14948804,cb910e924e2d,NaN,False,NaN,Sant Martí,el Camp de l'Arpa del Clot,4140698000,218241000,Private room in rental unit,...,27.0,472.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{""date"": 1614556800000, ""available_days"": 31,..."
1,1175393002332902164,b83bba1b76d4,False,False,True,Eixample,el Fort Pienc,4139989000,218471000,Private room in rental unit,...,58.0,459.0,True,24113.0,155.0,144.7,420.0,0.033,4.4,"[{""date"": 1717200000000, ""available_days"": 30,..."
2,1557316112282060644,3a5156e7345f,False,True,False,NaN,NaN,4137620000,216820000,Room in hotel,...,11.0,482.0,True,0.0,0.0,74.2,0.0,0.000,0.0,"[{""date"": 1761955200000, ""available_days"": 30,..."
3,3110443,88535ad25871,True,False,True,Eixample,la Dreta de l'Eixample,4139186000,217150000,Entire rental unit,...,172.0,470.0,True,155770.0,259.0,576.5,27098.0,0.622,283.5,"[{""date"": 1614556800000, ""available_days"": 24,..."
4,3491453,3c15645ab066,False,False,False,Eixample,la Sagrada Família,4140890000,217270000,Entire rental unit,...,142.0,454.0,True,79646.0,255.0,269.3,6660.0,0.222,65.9,"[{""date"": 1614556800000, ""available_days"": 31,..."


## 7.5 Data-type cleanup

Fix three known issues from the raw AirDNA export before any feature engineering:

- `latitude` and `longitude` are stored as integers scaled by **1e8** — rescale to decimal degrees so maps and distance calcs work.
- `star_rating` is stored as integer scaled by **100** — rescale to the 0-5 scale users expect.
- Add a `has_subdivision` flag so downstream analysis can filter out listings that can't be located at the neighbourhood level.
- Add `professional_management_known` so downstream can distinguish a *real* False from an *imputed-from-null* False.

In [17]:
# 1. Rescale latitude / longitude from integer (x1e8) to decimal degrees
for col in ("latitude", "longitude"):
    listings_clean[col] = listings_clean[col].astype(float) / 1e8

# 2. Rescale star_rating from integer (x100) to 0-5
listings_clean["star_rating"] = listings_clean["star_rating"].astype(float) / 100

# 3. Subdivision is the main geographic key — flag rows where it is missing
listings_clean["has_subdivision"] = listings_clean["subdivision"].notna()

# 4. Track whether professional_management was actually reported (vs imputed from null)
listings_clean["professional_management_known"] = listings_clean["professional_management"].notna()

print("latitude range  :", round(listings_clean["latitude"].min(), 4),
      "to", round(listings_clean["latitude"].max(), 4))
print("longitude range :", round(listings_clean["longitude"].min(), 4),
      "to", round(listings_clean["longitude"].max(), 4))
print("star_rating range:", listings_clean["star_rating"].min(),
      "to", listings_clean["star_rating"].max())
print("subdivision present:", int(listings_clean["has_subdivision"].sum()),
      "of", len(listings_clean))
print("professional_management known:", int(listings_clean["professional_management_known"].sum()),
      "of", len(listings_clean))

latitude range  : 41.353 to 41.4558
longitude range : 2.0856 to 2.2194
star_rating range: 1.0 to 5.0
subdivision present: 2354 of 2594
professional_management known: 1355 of 2594


## 8. Create listing-level features

These features will be useful for the risk score, clustering and policy analysis.


In [18]:
# Count how many listings each host manages
host_listing_count = listings_clean.groupby("host_id")["listing_id"].transform("nunique")

listings_clean["host_listing_count"] = host_listing_count
listings_clean["multi_listing_host"] = listings_clean["host_listing_count"] > 1
listings_clean["host_5_plus_listings"] = listings_clean["host_listing_count"] >= 5
listings_clean["host_10_plus_listings"] = listings_clean["host_listing_count"] >= 10

# Entire-home flag (explicit equality on the normalised room_type)
listings_clean["entire_home_flag"] = (
    listings_clean["room_type"].astype(str).str.lower() == "entire_home"
)

# Professional management flag.
# Keep original value, but create a boolean flag where available.
listings_clean["professional_management_flag"] = (
    listings_clean["professional_management"]
    .fillna(False)
    .astype(bool)
)

# Registration / licence availability flag.
listings_clean["has_registration"] = listings_clean["registration"].notna()

listings_clean.head()

,listing_id,host_id,professional_management,superhost,cohost,neighborhood,subdivision,latitude,longitude,listing_type,...,months,has_subdivision,professional_management_known,host_listing_count,multi_listing_host,host_5_plus_listings,host_10_plus_listings,entire_home_flag,professional_management_flag,has_registration
0,14948804,cb910e924e2d,NaN,False,NaN,Sant Martí,el Camp de l'Arpa del Clot,41.40698,2.18241,Private room in rental unit,...,"[{""date"": 1614556800000, ""available_days"": 31,...",True,False,1,False,False,False,False,False,False
1,1175393002332902164,b83bba1b76d4,False,False,True,Eixample,el Fort Pienc,41.39989,2.18471,Private room in rental unit,...,"[{""date"": 1717200000000, ""available_days"": 30,...",True,True,1,False,False,False,False,False,True
2,1557316112282060644,3a5156e7345f,False,True,False,NaN,NaN,41.37620,2.16820,Room in hotel,...,"[{""date"": 1761955200000, ""available_days"": 30,...",False,True,1,False,False,False,False,False,True
3,3110443,88535ad25871,True,False,True,Eixample,la Dreta de l'Eixample,41.39186,2.17150,Entire rental unit,...,"[{""date"": 1614556800000, ""available_days"": 24,...",True,True,2,True,False,False,True,True,True
4,3491453,3c15645ab066,False,False,False,Eixample,la Sagrada Família,41.40890,2.17270,Entire rental unit,...,"[{""date"": 1614556800000, ""available_days"": 31,...",True,True,1,False,False,False,True,False,True


## 9. Export listing-level clean dataset

In [19]:
listings_clean.to_csv(
    PROCESSED_DIR / "barcelona_listings_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "barcelona_listings_clean.csv")

Saved: ../data/processed/barcelona/barcelona_listings_clean.csv


## 10. Inspect embedded monthly history

The `months` column is a JSON string. Each row contains a list of monthly performance records for that listing.


In [20]:
sample_months = json.loads(listings.loc[0, "months"])

print("Type:", type(sample_months))
print("Number of months in first listing:", len(sample_months))
sample_months[0]

Type: <class 'list'>
Number of months in first listing: 38


{'date': 1614556800000,
 'available_days': 31,
 'unavailable_days': 0,
 'occupancy': 0,
 'rate_avg': 24.4,
 'native_rate_avg': 20,
 'revenue': 0,
 'native_revenue': 0,
 'active': False,
 'booking_lead_time_avg': None,
 'length_of_stay_avg': None,
 'booked_rate_avg': None,
 'native_booked_rate_avg': None,
 'rev_par': 0,
 'native_rev_par': 0}

In [21]:
sample_df = pd.DataFrame(sample_months)

sample_df["month_date"] = pd.to_datetime(
    sample_df["date"],
    unit="ms"
)

sample_df.head()

,date,available_days,unavailable_days,occupancy,rate_avg,native_rate_avg,revenue,native_revenue,active,booking_lead_time_avg,length_of_stay_avg,booked_rate_avg,native_booked_rate_avg,rev_par,native_rev_par,month_date
0,1614556800000,31,0,0.000,24.4,20,0,0,False,None,None,NaN,NaN,0.000000,0,2021-03-01
1,1617235200000,30,0,0.000,24.3,21,0,0,False,None,None,NaN,NaN,0.000000,0,2021-04-01
2,1619827200000,26,5,0.161,24.8,21,125,104,True,None,None,25.0,21.0,4.032258,3,2021-05-01
3,1622505600000,30,0,0.000,24.7,20,0,0,True,None,None,NaN,NaN,0.000000,0,2021-06-01
4,1625097600000,31,0,0.000,25.0,21,0,0,True,None,None,NaN,NaN,0.000000,0,2021-07-01


## 11. Expand monthly history for all listings

This transforms the embedded `months` JSON into a proper monthly panel dataset.

Final unit of analysis:

```text
1 row = 1 Airbnb listing + 1 month
```


In [22]:
monthly_records = []

for _, row in listings.iterrows():
    listing_id = row["listing_id"]

    if pd.isna(row["months"]):
        continue

    months_data = json.loads(row["months"])

    for month in months_data:
        month_record = month.copy()
        month_record["listing_id"] = listing_id
        monthly_records.append(month_record)

monthly_df = pd.DataFrame(monthly_records)

monthly_df["month_date"] = pd.to_datetime(
    monthly_df["date"],
    unit="ms"
)

print("Monthly dataset shape:", monthly_df.shape)
monthly_df.head()

Monthly dataset shape: (88821, 17)


,date,available_days,unavailable_days,occupancy,rate_avg,native_rate_avg,revenue,native_revenue,active,booking_lead_time_avg,length_of_stay_avg,booked_rate_avg,native_booked_rate_avg,rev_par,native_rev_par,listing_id,month_date
0,1614556800000,31,0,0.000,24.4,20,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-03-01
1,1617235200000,30,0,0.000,24.3,21,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-04-01
2,1619827200000,26,5,0.161,24.8,21,125,104,True,NaN,NaN,25.0,21.0,4.032258,3,14948804,2021-05-01
3,1622505600000,30,0,0.000,24.7,20,0,0,True,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-06-01
4,1625097600000,31,0,0.000,25.0,21,0,0,True,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-07-01


## 12. Monthly dataset audit

In [23]:
print("Rows:", len(monthly_df))
print("Unique listings:", monthly_df["listing_id"].nunique())
print("Min month:", monthly_df["month_date"].min())
print("Max month:", monthly_df["month_date"].max())

Rows: 88821
Unique listings: 2594
Min month: 2021-03-01 00:00:00
Max month: 2026-02-01 00:00:00


In [24]:
monthly_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88821 entries, 0 to 88820
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    88821 non-null  int64         
 1   available_days          88821 non-null  int64         
 2   unavailable_days        88821 non-null  int64         
 3   occupancy               88821 non-null  float64       
 4   rate_avg                88821 non-null  float64       
 5   native_rate_avg         88821 non-null  int64         
 6   revenue                 88821 non-null  int64         
 7   native_revenue          88821 non-null  int64         
 8   active                  88821 non-null  bool          
 9   booking_lead_time_avg   22574 non-null  float64       
 10  length_of_stay_avg      22574 non-null  float64       
 11  booked_rate_avg         50824 non-null  float64       
 12  native_booked_rate_avg  50824 non-null  float64       
 1

In [25]:
monthly_df.isnull().sum().sort_values(ascending=False)

booking_lead_time_avg     66247
length_of_stay_avg        66247
native_booked_rate_avg    37997
booked_rate_avg           37997
date                          0
listing_id                    0
native_rev_par                0
rev_par                       0
active                        0
available_days                0
native_revenue                0
revenue                       0
native_rate_avg               0
rate_avg                      0
occupancy                     0
unavailable_days              0
month_date                    0
dtype: int64

## 13. Clean monthly metrics

Rename the main pricing and revenue columns for clarity:

- `rate_avg` → `avg_daily_rate`
- `rev_par` → `revpar`


In [26]:
monthly_clean = monthly_df.rename(
    columns={
        "rate_avg": "avg_daily_rate",
        "rev_par": "revpar"
    }
)

monthly_clean.head()

,date,available_days,unavailable_days,occupancy,avg_daily_rate,native_rate_avg,revenue,native_revenue,active,booking_lead_time_avg,length_of_stay_avg,booked_rate_avg,native_booked_rate_avg,revpar,native_rev_par,listing_id,month_date
0,1614556800000,31,0,0.000,24.4,20,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-03-01
1,1617235200000,30,0,0.000,24.3,21,0,0,False,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-04-01
2,1619827200000,26,5,0.161,24.8,21,125,104,True,NaN,NaN,25.0,21.0,4.032258,3,14948804,2021-05-01
3,1622505600000,30,0,0.000,24.7,20,0,0,True,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-06-01
4,1625097600000,31,0,0.000,25.0,21,0,0,True,NaN,NaN,NaN,NaN,0.000000,0,14948804,2021-07-01


## 14. Export monthly clean dataset

In [27]:
monthly_clean.to_csv(
    PROCESSED_DIR / "barcelona_monthly_metrics_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "barcelona_monthly_metrics_clean.csv")

Saved: ../data/processed/barcelona/barcelona_monthly_metrics_clean.csv


## 15. Final outputs summary

This notebook creates two processed Barcelona datasets:

### `barcelona_listings_clean.csv`

Unit of analysis:

```text
1 row = 1 Airbnb listing
```

Use for:

- STR density,
- entire-home share,
- host concentration,
- professional management analysis,
- listing-level risk features,
- clustering inputs.

### `barcelona_monthly_metrics_clean.csv`

Unit of analysis:

```text
1 row = 1 Airbnb listing + 1 month
```

Coverage:

```text
March 2021 to February 2026
```

Use for:

- monthly occupancy,
- average daily rate,
- revenue,
- RevPAR,
- historical trends,
- emerging hotspot detection.
